# Bonus — No-Positional-Embeddings Ablation (Model H / Hindi)

Language Models and Agents, Monsoon 2026 — Individual Project.
Optional bonus (spec p. 12): pick one language, retrain with positional
embeddings removed, run the full Phase 2 evaluation suite, compare to the
standard model. **Chosen: Hindi (Model H)** — see `utility/bonus_plan.md`.

This is a **separate** run from `notebooks/phase2_kaggle.ipynb`: same code
and data datasets, but a different config
(`hindi/configs/hindi_model_ablation.yaml`, `positional: none`) writing to
an isolated output tree (`bonus_ablation/hindi/...`) that never overlaps
with the graded `hindi/checkpoints/`.

**Before running, in the sidebar:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Persistence | **Files only** |
| Input | `lma-code`, `lma-data-hindi` (no `lma-data-nepali` needed) |

Everything else about the architecture, optimizer, schedule, and corpus is
byte-identical to standard Model H — this isolates positional information
as the single ablated variable. Expect ~2-3 h to train, same order as the
standard Hindi run.


## Cell 1 — Environment

Same as `phase2_kaggle.ipynb`: Kaggle mounts datasets read-only, so the
code is copied somewhere writable and the corpus is **symlinked** rather
than copied.


In [ ]:
import os, shutil, subprocess, sys, time

REPO = '/kaggle/working/LMA_IDV_Proj'
MARKER = os.path.join('shared', 'model', 'transformer.py')


def find_dir_containing(marker, roots=('/kaggle/input',), max_depth=6):
    """Find the directory holding *marker*, wherever Kaggle mounted it.

    Mount layout differs by how the notebook was created: the UI gives a
    flat /kaggle/input/<slug>/, while an API-pushed notebook nests them
    under /kaggle/input/datasets/<owner>/<slug>/. Searching for a marker
    file works under both instead of hardcoding either.
    """
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _files in os.walk(root):
            if dirpath.count(os.sep) - root.count(os.sep) > max_depth:
                dirnames[:] = []
                continue
            if os.path.exists(os.path.join(dirpath, marker)):
                return dirpath
    return None


src_root = find_dir_containing(MARKER)
if src_root is None:
    print('Could not find the code dataset. What IS mounted:')
    for dirpath, dirnames, files in os.walk('/kaggle/input'):
        if dirpath.count(os.sep) <= 5:
            print(' ', dirpath, sorted(dirnames)[:6], sorted(files)[:4])
    raise FileNotFoundError('attach the lma-code dataset to this notebook')

print('code   :', src_root)
if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(src_root, REPO)
sys.path.insert(0, REPO)
os.chdir(REPO)
print('repo   :', REPO)


In [ ]:
def link_corpus(lang):
    """Symlink one language's tokenized .bin files AND held-out .txt splits
    into the paths its config expects. Same corpus as standard Model H --
    the ablation trains on identical data, only the model differs.
    """
    def link_dir(src, dst):
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            link = os.path.join(dst, f)
            if os.path.islink(link) or os.path.exists(link):
                os.remove(link) if os.path.islink(link) else None
            if not os.path.exists(link):
                os.symlink(os.path.join(src, f), link)

    bin_src = None
    txt_src = None
    for dirpath, _dirs, files in os.walk('/kaggle/input'):
        norm = dirpath.replace(os.sep, '/')
        if 'train.bin' in files and f'{lang}/data/tokenized' in norm:
            bin_src = dirpath
        if 'validation.txt' in files and f'{lang}/data/final' in norm:
            txt_src = dirpath
    if bin_src is None:
        print(f'  {lang}: not mounted — skipping')
        return False

    link_dir(bin_src, f'{REPO}/{lang}/data/tokenized')
    n = os.path.getsize(os.path.join(bin_src, 'train.bin')) // 4   # uint32
    print(f'  {lang}: {n:,} train tokens', end='')

    if txt_src is not None:
        link_dir(txt_src, f'{REPO}/{lang}/data/final')
        print(' | validation.txt + test.txt linked (BPB will use real bytes)')
    else:
        print(' | .txt splits not mounted — BPB falls back to config byte_counts')
    return True


print('corpora:')
link_corpus('hindi')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'sacrebleu', 'rouge-score'], check=False)

# Kaggle's font set differs from a local Linux box, and matplotlib's
# default has no Devanagari glyphs — it drops them with only a warning,
# which would silently ruin every attention heatmap.
subprocess.run('apt-get install -qq -y fonts-lohit-deva fonts-indic',
               shell=True, check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import torch
print('torch :', torch.__version__)
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    p = torch.cuda.get_device_properties(0)
    print(f'GPU   : {n} x {p.name}  ({p.total_memory / 1e9:.1f} GB each)')
else:
    print('GPU   : NONE — set Accelerator to GPU T4 x2 in the sidebar')


## Cell 2 — The ablation config

No language toggle here (unlike Phase 2's notebook) — this run is always
Hindi, always the ablated config. `positional: none` is the only model
change from `hindi_model.yaml`; every output path is routed under
`bonus_ablation/hindi/` so this can never collide with the graded
`hindi/checkpoints/`.


In [ ]:
LANGUAGE = 'hindi'
CONFIG = f'{LANGUAGE}/configs/{LANGUAGE}_model_ablation.yaml'
CKPT_DIR = f'bonus_ablation/{LANGUAGE}/checkpoints'
BEST = f'{CKPT_DIR}/best.pt'
LATEST = f'{CKPT_DIR}/latest.pt'

import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

print(f"{cfg['model_label']}")
print(f"  positional : {cfg['model']['positional']}")
print(f"  steps      : {cfg['training']['max_steps']:,}")
print(f"  tokens/step: {cfg['training']['micro_batch_size'] * cfg['model']['block_size'] * cfg['training']['grad_accum_steps']:,}")
print(f"  checkpoints -> {CKPT_DIR}")


## Cell 3 — Correctness gates

Same discipline as Phase 2: run before spending GPU time. The test suite
(`tests/test_phase2.py`) is not Hindi-standard-config-specific, so it still
applies. `count_params.py` hardcodes the standard config path internally
(no `--config` flag), so the parameter check below calls the underlying
report function directly against the ablation config instead.

Expect **25,059,328** — standard Model H's 25,321,472 minus the
262,144-parameter learned positional table (512 x 512) that
`NoPositional` (`shared/model/positional.py`) removes entirely: it holds
no `nn.Parameter` of its own, it just returns zeros. That exact
difference (262,144, and nothing else) is itself a sanity check that this
is a clean single-variable ablation — every other component's parameter
count is unchanged from standard Model H.


In [ ]:
gate = subprocess.run([sys.executable, '-m', 'pytest',
                       'tests/test_phase2.py', '-q'],
                      capture_output=True, text=True)
print(gate.stdout[-1500:])
assert gate.returncode == 0, 'correctness gates FAILED — do not train'


In [ ]:
from pathlib import Path
from shared.train.runner import print_parameter_report

print_parameter_report(CONFIG, Path(REPO), expect=25_059_328)


In [ ]:
!python {LANGUAGE}/eval/verify_causal_masking.py --config {CONFIG}


## Cell 4 — Train the ablated model

Same schedule as standard Model H (13,183 steps, identical cosine
schedule/optimizer/seed) — only `positional: none` differs. Resumes
automatically from `bonus_ablation/hindi/checkpoints/latest.pt` if a
session dies; re-running this identical cell is the recovery procedure.

**Do not pass `--max-steps`** — same reasoning as Phase 2: it would change
the cosine schedule and this would no longer be the same ablation.

Expect validation loss to plateau well above standard Model H's: with no
positional signal, the model can only learn a bag-of-tokens-per-prefix-
length distribution (see `shared/model/positional.py`'s `NoPositional`
docstring). This is the expected, correct outcome — not something to
"fix".

If it dies with CUDA OOM, use the fallback cell below instead.


In [ ]:
t0 = time.time()
!python {LANGUAGE}/train/pretrain.py --config {CONFIG}
print(f'\nwall clock: {(time.time() - t0) / 3600:.2f} h')


In [ ]:
# OOM fallback — same effective batch (16 x 512 x 6 = 49,152 tokens/step).
# Uncomment and run INSTEAD of the cell above.
# !python {LANGUAGE}/train/pretrain.py --config {CONFIG} --micro-batch-size 16 --grad-accum-steps 6


## Cell 5 — Back up the checkpoint

**Run this every session.** `/kaggle/working` is not a backup. Download
the zip from the Output tab, put it on Google Drive, and paste the link
into the README next to the standard Model H checkpoint link.


In [ ]:
stamp = time.strftime('%Y%m%d_%H%M')
out = f'/kaggle/working/hindi_ablation_{stamp}.zip'
!cd {REPO} && zip -q -1 -r {out} bonus_ablation/hindi/checkpoints bonus_ablation/hindi/logs
print(out, f'{os.path.getsize(out) / 1e6:.0f} MB')
print('-> download from the Output tab, then upload to Drive')


## Cell 6 — Evaluate (full Phase 2 suite, per spec p. 12)

Produces every metric the bonus requires: CE/PPL/BPB, generation at greedy
and T=0.5/1.0/1.5 scored with BLEU-4/chrF/ROUGE-L/Distinct-1,2/repetition,
and attention entropy, mean distance, and heatmaps — run against the
**ablated** checkpoint, for later side-by-side comparison with the
standard Model H results already in `hindi/eval/results/`.

Uses `best.pt` (lowest validation loss), not `latest.pt`.


In [ ]:
ckpt = BEST if os.path.exists(BEST) else LATEST
assert os.path.exists(ckpt), 'no checkpoint — run Cell 4 first'
print('evaluating', ckpt)

!python {LANGUAGE}/eval/evaluate_lm.py --config {CONFIG} --checkpoint {ckpt} --splits validation test
!python {LANGUAGE}/eval/evaluate_generation.py --config {CONFIG} --checkpoint {ckpt}
!python {LANGUAGE}/eval/analyze_attention.py --config {CONFIG} --checkpoint {ckpt} --device cpu


In [ ]:
# Confirm Devanagari actually rendered — matplotlib only WARNS about
# missing glyphs, so a heatmap of empty boxes otherwise looks like success.
from shared.eval import plots
font = plots.configure_devanagari_font()
ok = plots.verify_devanagari_rendering()
print(f'font: {font}  renders Devanagari: {ok}')
if not ok:
    print('WARNING: re-run the apt-get in Cell 1, then this cell')


## Cell 7 — Collect the graded artifacts

Same rule as Phase 2: *"Graders will not open external dashboards or
Drive folders to find figures; if it is not in the branch, it is not
graded."* Download this zip and **commit its contents to the `bonus`
branch** — figures, metric JSON, and training logs belong in git. Only
the checkpoint stays on Drive.


In [ ]:
art = f'/kaggle/working/hindi_ablation_artifacts_{stamp}.zip'
targets = f'bonus_ablation/hindi/figures bonus_ablation/hindi/eval_results bonus_ablation/hindi/logs'
!cd {REPO} && zip -q -r {art} {targets} -x '*.pt' '*.gitkeep'
print(art, f'{os.path.getsize(art) / 1e6:.1f} MB')
!cd {REPO} && unzip -l {art} | tail -25


## Cell 8 — Summary

After downloading and committing the artifacts (Cell 7) and uploading the
checkpoint to Drive (Cell 5), write the comparison in
`report/bonus_report.md`: ablated vs. standard Model H side by side (PPL,
BPB, BLEU/chrF/ROUGE-L, diversity, attention heatmaps/entropy/distance),
plus the explanation of what breaks without position information. See
`utility/bonus_plan.md` §4.4 for the required structure.


In [ ]:
import json

res = f'bonus_ablation/{LANGUAGE}/eval_results/lm_metrics.json'
if os.path.exists(res):
    with open(res) as f:
        m = json.load(f)['splits']['validation']
    print(f"{cfg['model_label']}")
    print(f"  CE loss    : {m['cross_entropy_nats']:.4f} nats/token")
    print(f"  perplexity : {m['perplexity']:.2f}")
    print(f"  BPB        : {m.get('bits_per_byte')}")
    print('\nCompare against standard Model H in hindi/eval/results/lm_metrics.json')
else:
    print('no LM metrics yet — run Cell 6')
